# 08 — Post-corrección de la clasificación española

Corrige un error sistemático detectado en la clasificación BART del corpus
español: la totalidad de los tweets sobre la «estafa del hijo en apuros»
(170 tweets) fueron asignados a la categoría `insurance`, cuando en realidad
constituyen un caso de suplantación de identidad y corresponden a
`phishing_identity`.

## Regla de corrección

Todo tweet clasificado como `insurance` cuyo texto contenga la expresión
«hijo en apuros» (o variantes) se reasigna a `phishing_identity`.

La regla es objetiva, transparente y reproducible. Se conserva una columna
`correccion_aplicada` para trazabilidad.

## Sobre payment_app

La infrarrepresentación de `payment_app` (BART no reconoce «Bizum») NO se
corrige: se documenta como limitación metodológica en la memoria. Este
notebook solo corrige el error sistemático y verificable del «hijo en apuros».

## Entrada / Salida

- Entrada: `scam_es_FINAL_classified.csv`
- Salida: `scam_es_FINAL_classified_corregido.csv`


In [ ]:
import pandas as pd
import re
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_es_FINAL_classified.csv")
print(f"Tweets cargados: {len(df)}")
print()
print("Distribución ANTES de la corrección:")
print(df["predicted_category"].value_counts())


## Aplicar la regla de corrección

In [ ]:
# Patrón de la estafa del hijo en apuros
PATRON_HIJO = re.compile(
    r"hij[oa]\s+en\s+apuros|"
    r"he\s+cambiado\s+de\s+n[uú]mero|"
    r"mam[aá].{0,30}nuevo\s+n[uú]mero|"
    r"este\s+es\s+mi\s+nuevo\s+n[uú]mero",
    re.IGNORECASE,
)

# Columna de trazabilidad
df["predicted_category_original"] = df["predicted_category"]
df["correccion_aplicada"] = ""

# Condición: clasificado como insurance Y el texto es de "hijo en apuros"
mask = (
    (df["predicted_category"] == "insurance") &
    (df["text"].fillna("").apply(lambda t: bool(PATRON_HIJO.search(t))))
)

n_corregidos = mask.sum()

df.loc[mask, "predicted_category"] = "phishing_identity"
df.loc[mask, "correccion_aplicada"] = "hijo_en_apuros: insurance -> phishing_identity"

print(f"Tweets reasignados de 'insurance' a 'phishing_identity': {n_corregidos}")
print()

# Cuántos quedan en insurance tras la corrección
insurance_restante = (df["predicted_category"] == "insurance").sum()
print(f"Tweets que permanecen en 'insurance': {insurance_restante}")
if insurance_restante > 0:
    print("\nLos que quedan en insurance (revisar que sí sean de seguros):")
    for _, r in df[df["predicted_category"]=="insurance"].iterrows():
        print(f"  [{r['confidence_score']:.2f}] {str(r['text'])[:180]}")


## Distribución después de la corrección

In [ ]:
print("Distribución DESPUÉS de la corrección:\n")
counts = df["predicted_category"].value_counts()
total = len(df)
for cat, n in counts.items():
    print(f"  {cat:<22} {n:>5}  ({n/total*100:>5.1f}%)")
print(f"\nRelevantes: {(df['predicted_category']!='not_related').sum()} / {total}")


In [ ]:
# Recalcular is_relevant por si acaso
df["is_relevant"] = df["predicted_category"] != "not_related"

# Comparativa antes/después
print("=== COMPARATIVA ANTES / DESPUÉS ===\n")
antes = df["predicted_category_original"].value_counts()
despues = df["predicted_category"].value_counts()
cats = sorted(set(antes.index) | set(despues.index))
print(f"  {'categoría':<22} {'antes':>7} {'después':>8}")
print("  " + "-"*40)
for c in cats:
    a = antes.get(c, 0)
    d = despues.get(c, 0)
    marca = "  <--" if a != d else ""
    print(f"  {c:<22} {a:>7} {d:>8}{marca}")


## Guardado del CSV corregido

In [ ]:
OUTPUT = "../data/raw/scam_es_FINAL_classified_corregido.csv"
df.to_csv(OUTPUT, index=False, encoding="utf-8")
print(f"✓ Guardado: {OUTPUT}")
print(f"  {len(df)} tweets")
print()
print("Columnas de trazabilidad añadidas:")
print("  predicted_category_original  -> categoría antes de corregir")
print("  predicted_category           -> categoría definitiva (USAR ESTA)")
print("  correccion_aplicada          -> marca de los tweets corregidos")
print()
print("📦 scam_es_FINAL_classified_corregido.csv = dataset español DEFINITIVO")
print("🚨 Backup en Drive.")


## Texto para la memoria

In [ ]:
n_corr = (df["correccion_aplicada"] != "").sum()
print(f"""
PARA LA MEMORIA — post-corrección de la clasificación española
{'='*60}

Durante la validación de la clasificación del corpus español se
detectó un error sistemático: la totalidad de los {n_corr} tweets
relativos a la «estafa del hijo en apuros» fueron asignados por el
modelo BART a la categoría 'fraude de seguros' (insurance). Se
atribuye este comportamiento a la dificultad del modelo, entrenado
mayoritariamente en inglés, para procesar correctamente esta
tipología expresada en español.

Dado que la «estafa del hijo en apuros» constituye un caso de
suplantación de identidad mediante ingeniería social, se aplicó una
regla de post-corrección, objetiva y reproducible, que reasigna
dichos tweets a la categoría 'phishing o robo de identidad'
(phishing_identity), de forma coherente con la tipología de INCIBE
y con el tratamiento de los casos equivalentes en el corpus
estadounidense.

Asimismo, se observó que la categoría 'estafa en aplicaciones de
pago' (payment_app) queda infrarrepresentada en el corpus español:
el modelo BART no reconoce el término «Bizum» como referente de
dicha categoría, distribuyendo esos tweets entre categorías afines.
Esta limitación, derivada del uso de un modelo anglófono sobre texto
en español, se documenta y debe tenerse en cuenta al interpretar la
frecuencia de esa categoría concreta.
{'='*60}
""")
